In [2]:
"""
03_revenue_forecast.py
========================
Answers business question 3: What is next quarter's revenue forecast?

Approach:
  - Aggregate WON deal value into a monthly revenue time series.
  - Fit a Holt-Winters (triple exponential smoothing) model, which
    captures trend + yearly seasonality -- a good, lightweight fit for
    3 years of monthly data (36 points). This mirrors what ML.NET's
    ForecastBySsa does on the .NET path.
  - Forecast the next 3 months and attach 80%/95% confidence intervals
    via residual-based simulation (statsmodels' built-in simulation),
    so leadership sees a RANGE, not a single fragile number
    (see "Confidence intervals on forecasts" enhancement in the brief).

Outputs:
  - monthly_revenue_forecast.png
  - revenue_forecast_next_3_months.csv
  - revenue_forecast_model.pkl
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pickle

from statsmodels.tsa.holtwinters import ExponentialSmoothing

df = pd.read_csv("synthetic_crm_data.csv")
won = df[df["is_won"] == 1].copy()
won["close_date"] = pd.to_datetime(won["close_date"])

monthly = (won.set_index("close_date")
              .resample("MS")["close_value"]
              .sum())
monthly = monthly.asfreq("MS", fill_value=0)  # ensure no gaps

print("=== Monthly revenue (won deals) ===")
print(monthly.round(0))

# The most recent 1-2 months are always artificially low in ANY live CRM
# extract: deals engaged recently haven't had time to close yet
# ("right-censoring" / pipeline lag). Real sales-ops teams exclude these
# incomplete trailing months before fitting a trend model, otherwise the
# forecast wrongly reads a real dip as "revenue is declining."
INCOMPLETE_TRAILING_MONTHS = 2
monthly_for_fit = monthly.iloc[:-INCOMPLETE_TRAILING_MONTHS]
print(f"\n(Excluding last {INCOMPLETE_TRAILING_MONTHS} month(s) from model fitting "
      f"as incomplete/still-closing pipeline)")

# ---------------------------------------------------------------------
# Fit Holt-Winters: additive trend, additive yearly seasonality
# ---------------------------------------------------------------------
model = ExponentialSmoothing(
    monthly_for_fit, trend="add", seasonal="add", seasonal_periods=12,
    initialization_method="estimated"
).fit(optimized=True)

FORECAST_MONTHS = 3
# Forecast far enough to (a) "complete" the trailing incomplete months
# still finishing in the pipeline, then (b) give 3 genuinely-future months
# beyond the last real data point.
TOTAL_HORIZON = INCOMPLETE_TRAILING_MONTHS + FORECAST_MONTHS
forecast = model.forecast(TOTAL_HORIZON)

# Confidence intervals via simulation (statsmodels supports this natively
# for ExponentialSmoothing results)
n_sims = 2000
sims = model.simulate(TOTAL_HORIZON, repetitions=n_sims, error="add",
                       random_state=42)
lower_80 = sims.quantile(0.10, axis=1)
upper_80 = sims.quantile(0.90, axis=1)
lower_95 = sims.quantile(0.025, axis=1)
upper_95 = sims.quantile(0.975, axis=1)

full_bridge_df = pd.DataFrame({
    "month": forecast.index,
    "forecast_revenue": forecast.values.round(0),
    "lower_80": lower_80.values.round(0),
    "upper_80": upper_80.values.round(0),
    "lower_95": lower_95.values.round(0),
    "upper_95": upper_95.values.round(0),
    "type": (["pipeline_completion_estimate"] * INCOMPLETE_TRAILING_MONTHS
              + ["forward_forecast"] * FORECAST_MONTHS),
})
print("\n=== Forecast bridge: completing the pipeline + next 3 forward months ===")
print(full_bridge_df.to_string(index=False))

forecast_df = full_bridge_df[full_bridge_df["type"] == "forward_forecast"].drop(columns="type").reset_index(drop=True)
print("\n=== Next 3 (forward-looking) months revenue forecast ===")
print(forecast_df.to_string(index=False))
forecast_df.to_csv("revenue_forecast_next_3_months.csv", index=False)
full_bridge_df.to_csv("revenue_forecast_full_bridge.csv", index=False)

# ---------------------------------------------------------------------
# Simple backtest: hide last 3 real months, forecast them, compare
# (gives you an honest error metric to quote instead of just "trust me")
# ---------------------------------------------------------------------
train = monthly_for_fit.iloc[:-3]
test = monthly_for_fit.iloc[-3:]
bt_model = ExponentialSmoothing(
    train, trend="add", seasonal="add", seasonal_periods=12,
    initialization_method="estimated"
).fit(optimized=True)
bt_forecast = bt_model.forecast(3)
mape = float(np.mean(np.abs((test.values - bt_forecast.values) / np.where(test.values == 0, 1, test.values))) * 100)
print(f"\n=== Backtest: forecasting the last known 3 months ===")
print(pd.DataFrame({"actual": test.values, "predicted": bt_forecast.values.round(0)}, index=test.index))
print(f"MAPE (mean absolute % error): {mape:.1f}%")

# ---------------------------------------------------------------------
# Plot
# ---------------------------------------------------------------------
plt.figure(figsize=(12, 6))
plt.plot(monthly_for_fit.index, monthly_for_fit.values, label="Actual monthly revenue (used for fitting)",
          marker="o", color="steelblue")
plt.plot(monthly.index[-(INCOMPLETE_TRAILING_MONTHS + 1):], monthly.values[-(INCOMPLETE_TRAILING_MONTHS + 1):],
          label="Incomplete trailing months (excluded)", marker="o", linestyle=":", color="gray")
plt.plot(forecast_df["month"], forecast_df["forecast_revenue"], label="Forecast", marker="o",
         linestyle="--", color="darkorange")
plt.fill_between(forecast_df["month"], forecast_df["lower_95"], forecast_df["upper_95"],
                  alpha=0.15, color="darkorange", label="95% confidence interval")
plt.fill_between(forecast_df["month"], forecast_df["lower_80"], forecast_df["upper_80"],
                  alpha=0.3, color="darkorange", label="80% confidence interval")
plt.title(f"Monthly Revenue Forecast (next {FORECAST_MONTHS} months) — backtest MAPE {mape:.1f}%")
plt.xlabel("Month")
plt.ylabel("Revenue ($)")
plt.legend()
plt.tight_layout()
plt.savefig("monthly_revenue_forecast.png", dpi=150)
plt.close()
print("Saved -> monthly_revenue_forecast.png")

with open("revenue_forecast_model.pkl", "wb") as f:
    pickle.dump(model, f)
print("Saved -> revenue_forecast_model.pkl")

=== Monthly revenue (won deals) ===
close_date
2022-01-01       6760.0
2022-02-01      90202.0
2022-03-01     259136.0
2022-04-01     622605.0
2022-05-01     984879.0
2022-06-01    1018250.0
2022-07-01    1168525.0
2022-08-01    1034246.0
2022-09-01    1235216.0
2022-10-01    1186343.0
2022-11-01    1321555.0
2022-12-01    1346872.0
2023-01-01    1437126.0
2023-02-01    1392403.0
2023-03-01    1539037.0
2023-04-01    1390926.0
2023-05-01    1410561.0
2023-06-01    1430275.0
2023-07-01    1359221.0
2023-08-01    1566410.0
2023-09-01    1510672.0
2023-10-01    1468630.0
2023-11-01    1592964.0
2023-12-01    1643034.0
2024-01-01    1699165.0
2024-02-01    1701180.0
2024-03-01    1771454.0
2024-04-01    1752658.0
2024-05-01    1753746.0
2024-06-01    1855304.0
2024-07-01    1728677.0
2024-08-01    1632266.0
2024-09-01    1540646.0
2024-10-01    1394636.0
2024-11-01    1250638.0
2024-12-01     986596.0
Freq: MS, Name: close_value, dtype: float64

(Excluding last 2 month(s) from model fittin

In [3]:
"""
02_winrate_and_cycle_analysis.py
==================================
Answers business questions 1 and 2 from the playbook:
  Q1: What factors most influence win rate?      -> correlation + feature importance
  Q2: How do sales cycles vary by industry?       -> descriptive stats + boxplot

Trains a RandomForestClassifier to predict is_won (only for CLOSED deals),
using features that are known at decision time (no leakage from close_value,
which is 0/blank until a deal is already decided).

Outputs:
  - winrate_feature_importance.png
  - sales_cycle_by_sector_boxplot.png
  - correlation_heatmap.png
  - win_rate_model.pkl          (trained classifier)
  - win_rate_model_columns.pkl  (feature/column metadata needed to reuse the model)
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.preprocessing import OneHotEncoder

sns.set_theme(style="whitegrid")

df = pd.read_csv("synthetic_crm_data.csv")
closed = df[df["is_closed"] == 1].copy()

# ---------------------------------------------------------------------
# Q2: Sales cycle length by industry (descriptive stats + boxplot)
# ---------------------------------------------------------------------
cycle_stats = (closed.groupby("sector")["sales_cycle_days"]
               .agg(["count", "mean", "median", "std", "min", "max"])
               .sort_values("mean", ascending=False)
               .round(1))
print("=== Sales cycle length by sector (days) ===")
print(cycle_stats)
cycle_stats.to_csv("sales_cycle_by_sector_stats.csv")

plt.figure(figsize=(11, 6))
order = closed.groupby("sector")["sales_cycle_days"].median().sort_values(ascending=False).index
sns.boxplot(data=closed, x="sector", y="sales_cycle_days", order=order, palette="viridis")
plt.xticks(rotation=40, ha="right")
plt.title("Sales Cycle Length by Industry Sector")
plt.ylabel("Sales cycle (days)")
plt.xlabel("")
plt.tight_layout()
plt.savefig("sales_cycle_by_sector_boxplot.png", dpi=150)
plt.close()
print("Saved -> sales_cycle_by_sector_boxplot.png")

# ---------------------------------------------------------------------
# Q1: What drives win rate? -- correlation heatmap on numeric fields
# ---------------------------------------------------------------------
numeric_cols = ["deal_value_proposed", "revenue", "employees", "sales_cycle_days", "is_won"]
corr = closed[numeric_cols].corr()
print("\n=== Correlation with is_won ===")
print(corr["is_won"].sort_values(ascending=False))

plt.figure(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap (numeric fields)")
plt.tight_layout()
plt.savefig("correlation_heatmap.png", dpi=150)
plt.close()
print("Saved -> correlation_heatmap.png")

# ---------------------------------------------------------------------
# Q1 continued: RandomForest classifier -> feature importance
# (importance captures nonlinear + categorical effects that a plain
#  correlation matrix misses, e.g. sector, product, agent, quarter)
# ---------------------------------------------------------------------
closed["engage_month"] = pd.to_datetime(closed["engage_date"]).dt.month
closed["engage_quarter"] = pd.to_datetime(closed["engage_date"]).dt.quarter

feature_cols_num = ["deal_value_proposed", "revenue", "employees",
                     "sales_cycle_days", "engage_month", "engage_quarter"]
feature_cols_cat = ["sector", "product", "regional_office", "office_location", "sales_agent"]

X_num = closed[feature_cols_num].fillna(closed[feature_cols_num].median())
enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
X_cat = enc.fit_transform(closed[feature_cols_cat])
cat_feature_names = enc.get_feature_names_out(feature_cols_cat)

X = np.hstack([X_num.values, X_cat])
all_feature_names = feature_cols_num + list(cat_feature_names)
y = closed["is_won"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

clf = RandomForestClassifier(n_estimators=400, max_depth=14, min_samples_leaf=8,
                              class_weight="balanced_subsample",
                              random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

y_prob = clf.predict_proba(X_test)[:, 1]
# Use 0.5 threshold on balanced-weighted model; report AUC as the primary
# metric since raw accuracy is a poor measure under class imbalance.
y_pred = (y_prob >= 0.5).astype(int)
auc = roc_auc_score(y_test, y_prob)
print(f"\n=== Win-rate classifier performance ===\nROC-AUC: {auc:.3f}  (0.5=random, 1.0=perfect)")
print(classification_report(y_test, y_pred, target_names=["Lost", "Won"]))

importances = pd.Series(clf.feature_importances_, index=all_feature_names)

# Roll up one-hot categorical importances back to their parent field
rollup = {}
for name, val in importances.items():
    parent = name
    for cat_col in feature_cols_cat:
        if name.startswith(cat_col + "_"):
            parent = cat_col
            break
    rollup[parent] = rollup.get(parent, 0) + val
rollup = pd.Series(rollup).sort_values(ascending=False)

print("\n=== What most influences win rate (feature importance, rolled up) ===")
print(rollup.round(3))

plt.figure(figsize=(8, 5))
rollup.sort_values().plot(kind="barh", color="steelblue")
plt.title("What Most Influences Win Rate (RandomForest feature importance)")
plt.xlabel("Relative importance")
plt.tight_layout()
plt.savefig("winrate_feature_importance.png", dpi=150)
plt.close()
print("Saved -> winrate_feature_importance.png")

with open("win_rate_model.pkl", "wb") as f:
    pickle.dump(clf, f)
with open("win_rate_model_columns.pkl", "wb") as f:
    pickle.dump({
        "feature_cols_num": feature_cols_num,
        "feature_cols_cat": feature_cols_cat,
        "encoder": enc,
        "all_feature_names": all_feature_names,
    }, f)
print("Saved -> win_rate_model.pkl, win_rate_model_columns.pkl")

=== Sales cycle length by sector (days) ===
                    count   mean  median   std   min    max
sector                                                     
medical              2742  109.9   108.0  25.5  34.0  217.0
finance              3065  104.3   103.0  25.0  23.0  191.0
telecommunications   2180   94.5    94.0  23.8  21.0  178.0
technology           5373   84.8    84.0  23.3  14.0  174.0
software             4337   78.6    78.0  22.2  11.0  168.0
entertainment        2012   74.5    74.0  22.3  11.0  168.0
Unassigned Sector    1153   69.9    69.0  21.9   5.0  143.0
services             4423   69.7    69.0  21.9   5.0  172.0
marketing            2770   64.5    63.0  21.0   5.0  144.0
retail               3491   60.1    59.0  20.7   5.0  150.0
employment           2517   54.2    54.0  20.5   5.0  127.0


/tmp/ipykernel_2028/517593859.py:51: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=closed, x="sector", y="sales_cycle_days", order=order, palette="viridis")


Saved -> sales_cycle_by_sector_boxplot.png

=== Correlation with is_won ===
is_won                 1.000000
revenue                0.002426
employees             -0.005200
deal_value_proposed   -0.109337
sales_cycle_days      -0.134848
Name: is_won, dtype: float64
Saved -> correlation_heatmap.png

=== Win-rate classifier performance ===
ROC-AUC: 0.657  (0.5=random, 1.0=perfect)
              precision    recall  f1-score   support

        Lost       0.71      0.59      0.64      4129
         Won       0.50      0.64      0.56      2684

    accuracy                           0.61      6813
   macro avg       0.61      0.61      0.60      6813
weighted avg       0.63      0.61      0.61      6813


=== What most influences win rate (feature importance, rolled up) ===
sales_agent            0.342
sales_cycle_days       0.141
deal_value_proposed    0.136
product                0.074
sector                 0.062
revenue                0.060
employees              0.060
office_location   

In [ ]:
"""
train_models.py
----------------
Trains three models from sales_data.csv:

 1. revenue_forecast_model   -> forecasts next 3 months of company revenue
    (time-series, Holt-Winters exponential smoothing + a GradientBoosting
    lag-feature model as a second opinion)

 2. employee_next_quarter_model -> predicts a rep's NEXT quarter revenue
    from their trailing performance stats (this is the "employee
    performance" model)

 3. employee_future_years_model -> predicts a rep's revenue trend for
    future years, used to project 1-3 years ahead per rep

All three are saved with joblib into ../saved_models/ so the dashboard
(or the Ollama Q&A tool) can load them without retraining.

Run:  python3 train_models.py
"""

import numpy as np
import pandas as pd
import joblib
import os
import warnings
warnings.filterwarnings("ignore")

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from statsmodels.tsa.holtwinters import ExponentialSmoothing

DATA_PATH = "/home/claude/sales_intelligence/sales_data.csv"
OUT_DIR = "/home/claude/sales_intelligence/saved_models"
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH, parse_dates=["created_date"])
df["close_date"] = pd.to_datetime(df["close_date"], errors="coerce")
won = df[df["is_won"] == 1].copy()

# ============================================================
# MODEL 1: Company-wide revenue forecast (next 3 months)
# ============================================================
print("=" * 60)
print("MODEL 1: Revenue forecast (next 3 months)")
print("=" * 60)

won["close_month"] = won["close_date"].dt.to_period("M")
monthly_revenue_full = won.groupby("close_month")["revenue"].sum().sort_index()
monthly_revenue_full.index = monthly_revenue_full.index.to_timestamp()
monthly_revenue_full = monthly_revenue_full.asfreq("MS").fillna(0)

# Train only on the COMPLETE historical window (2023-01 .. 2025-12). Later
# months are just the natural tail of deals still in the pipeline that
# haven't closed yet as of the data snapshot, not a real trend -- including
# them would bias the forecast downward for no real reason.
monthly_revenue = monthly_revenue_full.loc[:"2025-12-01"]

print(f"Monthly revenue series: {len(monthly_revenue)} months "
      f"({monthly_revenue.index.min().date()} -> {monthly_revenue.index.max().date()})")

# Holt-Winters handles trend + yearly seasonality well for this kind of series.
# With only ~3 years of history, an additive seasonal + trend fit can occasionally
# extrapolate below zero, so we try seasonal first and fall back to a damped
# trend-only model if the forecast isn't sane, then floor at 0 either way.
def fit_forecast_safely(series, periods=3):
    try:
        model = ExponentialSmoothing(
            series, trend="add", damped_trend=True, seasonal="add",
            seasonal_periods=12, initialization_method="estimated"
        ).fit()
        fc = model.forecast(periods)
        if (fc < 0).any():
            raise ValueError("seasonal fit produced negative forecast, falling back")
        return model, fc
    except Exception:
        model = ExponentialSmoothing(
            series, trend="add", damped_trend=True, seasonal=None,
            initialization_method="estimated"
        ).fit()
        fc = model.forecast(periods)
        return model, fc

hw_model, hw_forecast = fit_forecast_safely(monthly_revenue, 3)
hw_forecast = hw_forecast.clip(lower=0)
print("\nHolt-Winters 3-month forecast:")
print(hw_forecast.round(2))

# Lag-feature GradientBoosting model as a cross-check / alternative forecaster
def build_lag_features(series, n_lags=6):
    frame = pd.DataFrame({"y": series})
    for lag in range(1, n_lags + 1):
        frame[f"lag_{lag}"] = frame["y"].shift(lag)
    frame["month_of_year"] = frame.index.month
    frame = frame.dropna()
    return frame

lag_frame = build_lag_features(monthly_revenue, n_lags=6)
X_ts = lag_frame.drop(columns=["y"])
y_ts = lag_frame["y"]

gbr_ts = GradientBoostingRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, random_state=42)
gbr_ts.fit(X_ts, y_ts)

# Recursively forecast 3 months ahead with the lag model
history = monthly_revenue.copy()
gbr_preds = []
for step in range(3):
    last_row = {}
    for lag in range(1, 7):
        last_row[f"lag_{lag}"] = history.iloc[-lag]
    next_month = history.index[-1] + pd.DateOffset(months=1)
    last_row["month_of_year"] = next_month.month
    pred = gbr_ts.predict(pd.DataFrame([last_row]))[0]
    gbr_preds.append((next_month, pred))
    history.loc[next_month] = pred

print("\nGradientBoosting 3-month forecast (cross-check):")
for d, v in gbr_preds:
    print(f"  {d.date()}: {v:,.2f}")

joblib.dump(hw_model, os.path.join(OUT_DIR, "revenue_forecast_holtwinters.joblib"))
joblib.dump(gbr_ts, os.path.join(OUT_DIR, "revenue_forecast_gbr.joblib"))
joblib.dump(monthly_revenue, os.path.join(OUT_DIR, "monthly_revenue_series.joblib"))

# ============================================================
# MODEL 2: Employee next-quarter performance/revenue prediction
# ============================================================
print("\n" + "=" * 60)
print("MODEL 2: Employee next-quarter revenue (performance model)")
print("=" * 60)

won["close_quarter"] = won["close_date"].dt.to_period("Q")
df["created_quarter"] = df["created_date"].dt.to_period("Q")
won_for_panel = won[won["close_date"] <= "2025-12-31"]  # exclude partial pipeline tail

# Build a rep x quarter panel: revenue won, deals worked, win rate, avg cycle, avg deal size
rep_quarter_revenue = won_for_panel.groupby(["rep_id", "close_quarter"])["revenue"].sum().rename("revenue")
rep_quarter_deals = df.groupby(["rep_id", "created_quarter"]).agg(
    deals_worked=("opportunity_id", "count"),
    win_rate=("is_won", "mean"),
    avg_deal_size=("deal_size", "mean"),
    avg_cycle_days=("sales_cycle_days", "mean"),
    avg_activities=("num_activities", "mean"),
).rename_axis(index=["rep_id", "quarter"])

panel = rep_quarter_deals.join(
    rep_quarter_revenue.rename_axis(index=["rep_id", "quarter"]), how="left"
).fillna(0.0).reset_index()
panel = panel.sort_values(["rep_id", "quarter"])

# Features = this quarter's stats, Target = NEXT quarter's revenue for that rep
panel["next_quarter_revenue"] = panel.groupby("rep_id")["revenue"].shift(-1)
train_panel = panel.dropna(subset=["next_quarter_revenue"]).copy()

feature_cols = ["deals_worked", "win_rate", "avg_deal_size", "avg_cycle_days",
                 "avg_activities", "revenue"]
X_emp = train_panel[feature_cols]
y_emp = train_panel["next_quarter_revenue"]

X_train, X_test, y_train, y_test = train_test_split(X_emp, y_emp, test_size=0.2, random_state=42)

emp_model = RandomForestRegressor(n_estimators=400, max_depth=8, random_state=42, n_jobs=-1)
emp_model.fit(X_train, y_train)
pred = emp_model.predict(X_test)
print(f"MAE: {mean_absolute_error(y_test, pred):,.2f}")
print(f"R2:  {r2_score(y_test, pred):.3f}")
print("\nFeature importance:")
for f, imp in sorted(zip(feature_cols, emp_model.feature_importances_), key=lambda x: -x[1]):
    print(f"  {f}: {imp:.3f}")

joblib.dump(emp_model, os.path.join(OUT_DIR, "employee_next_quarter_model.joblib"))
joblib.dump(feature_cols, os.path.join(OUT_DIR, "employee_model_features.joblib"))
panel.to_csv(os.path.join(OUT_DIR, "rep_quarter_panel.csv"), index=False)

# ============================================================
# MODEL 3: Employee revenue in coming years (multi-year trend)
# ============================================================
print("\n" + "=" * 60)
print("MODEL 3: Employee revenue trend for future years")
print("=" * 60)

won["close_year"] = won["close_date"].dt.year
won_complete_years = won[won["close_year"] <= 2025]  # exclude partial 2026 pipeline tail
rep_year_revenue = won_complete_years.groupby(["rep_id", "close_year"])["revenue"].sum().reset_index()

# Merge rep tenure/skill context (tenure at first deal, deal count trend)
rep_first_year = df.groupby("rep_id")["created_date"].min().dt.year.rename("first_active_year")
rep_year_revenue = rep_year_revenue.merge(rep_first_year, on="rep_id", how="left")
rep_year_revenue["years_active"] = rep_year_revenue["close_year"] - rep_year_revenue["first_active_year"]

# Also bring in deal counts per rep-year for signal
rep_year_deals = df.groupby(["rep_id", df["created_date"].dt.year]).agg(
    deals_worked=("opportunity_id", "count"),
    win_rate=("is_won", "mean"),
).rename_axis(index=["rep_id", "close_year"]).reset_index()

year_panel = rep_year_revenue.merge(rep_year_deals, on=["rep_id", "close_year"], how="left").fillna(0)
year_panel = year_panel.sort_values(["rep_id", "close_year"])
year_panel["next_year_revenue"] = year_panel.groupby("rep_id")["revenue"].shift(-1)
train_year = year_panel.dropna(subset=["next_year_revenue"]).copy()

year_features = ["revenue", "years_active", "deals_worked", "win_rate"]
X_yr = train_year[year_features]
y_yr = train_year["next_year_revenue"]

Xy_train, Xy_test, yy_train, yy_test = train_test_split(X_yr, y_yr, test_size=0.2, random_state=42)
year_model = GradientBoostingRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, random_state=42)
year_model.fit(Xy_train, yy_train)
yr_pred = year_model.predict(Xy_test)
print(f"MAE: {mean_absolute_error(yy_test, yr_pred):,.2f}")
print(f"R2:  {r2_score(yy_test, yr_pred):.3f}")

joblib.dump(year_model, os.path.join(OUT_DIR, "employee_future_years_model.joblib"))
joblib.dump(year_features, os.path.join(OUT_DIR, "employee_year_model_features.joblib"))
year_panel.to_csv(os.path.join(OUT_DIR, "rep_year_panel.csv"), index=False)

print("\nAll models trained and saved to:", OUT_DIR)